# Function calling

https://platform.openai.com/docs/guides/function-calling

<img src="https://cdn.openai.com/API/docs/images/function-calling-diagram-steps.png" alt="Function Calling Diagram" width="600"/>

Function Calling은 모델이 직접 외부 API나 Python 함수를 실행하는 기능이 아니다.

모델은 다음 두 가지를 판단한다.

1. 어떤 함수를 호출해야 하는가
2. 함수에 어떤 인자를 전달해야 하는가

실제 함수 실행은 개발자의 코드에서 수행한다.

In [1]:
# 공통 설정
# .env 파일에서 API Key와 기본 모델명을 읽어온다.
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPEAI_API_KEY를 환경 변수로 설정하세요.")

client = OpenAI(api_key=api_key)
DEFAULT_MODEL = os.getenv("OPENAI_DEFAULT_MODEL", "gpt-4.1-mini")

print("OpenAI client 준비 완료")
print("기본 모델 : ", DEFAULT_MODEL)

OpenAI client 준비 완료
기본 모델 :  gpt-4.1-mini


## 수강생 상담용 함수
- 외부 API 없이 Function Calling 동작 구조를 확인
- 실제 서비스의 경우 DB조회, 통계 계산, 학습 이력 분석 로직 등으로 바뀔 수 있음

In [2]:
students = {
    "철수" : {"score" : 82, "attendance": 0.92, "late_count":1},
    "관순" : {"score" : 61, "attendance": 0.76, "late_count":4},
    "순신" : {"score" : 95, "attendance": 0.98, "late_count":0},
}

def make_learning_feedback(score,attendance,late_count):
    if score >= 90 and attendance >= 0.9:
        level = "우수"
        message = "전체 흐름이 좋으므로 심화 과제를 제공해도 좋습니다."
    elif score >= 70 and attendance >=0.85:
        level = "보통"
        message = "기본기는 있으나 오답 유형을 점검하는 보충 수업이 필요합니다."
    else:
        level = "관리 필요"
        message = "출석 기본 개념 복습을 함께 관리해야 합니다."

    if late_count >= 3:
        message += "지각 횟수가 너무 많으므로 학습 루틴 점검도 필요합니다."

    return {
        "level" : level,
        "message" : message
    }

def get_student_counseling(name):
    """학생 이름을 받아 상태 조회와 학습 피드백을 함께 수행한다."""
    data = students.get(name)

    if not data:
        return {
            "found" : False,
            "message" : f"{name} 학생 정보를 찾을 수 없습니다."
        }
    
    feedback = make_learning_feedback(
        score=data['score'],
        attendance=data['attendance'],
        late_count=data['late_count']
    )

    return {
        "found" : True,
        "name" : name,
        **data,
        **feedback
    }

print(get_student_counseling('철수'))
print(get_student_counseling('수진'))

{'found': True, 'name': '철수', 'score': 82, 'attendance': 0.92, 'late_count': 1, 'level': '보통', 'message': '기본기는 있으나 오답 유형을 점검하는 보충 수업이 필요합니다.'}
{'found': False, 'message': '수진 학생 정보를 찾을 수 없습니다.'}


## Tool Schema 작성
- 모델은 함수를 직접 볼 수 없다. 따라서 함수 이름, 설명, 파라미터 구조를 JSON Schema 형태로 알려줘야 한다.
- 모델은 이 설명을 보고 어떤 상황에서 어떤 함수를 호출할지 판단한다.

In [3]:
student_tools = [
    {
        "type" : "function",
        "name" : "get_student_counseling",
        "description" : "학생 이름을 받아 점수, 출석률, 지각 횟수와 학습 상담 피드백을 함께 조회한다.",
        "parameters" : {
            "type" : "object",
            "additionalProperties" : False,
            "properties" : {
                "name":{
                    "type" : "string",
                    "description" : "학생 이름"
                }
            },
            "required" : ["name"]
        }
    }
]

student_function_map = {
    "get_student_counseling" : get_student_counseling
}

## Function Calling 실행 헬퍼

In [6]:
import json

def run_function_call(item, function_map):
    """모델이 요청한 function_call item을 실제 함수 실행 결과로 변환한다."""

    name = item.name                        # 모델이 호출하겠다고 선택한 함수 이름
    args = json.loads(item.arguments)        # 모델이 생성한 함수 인자를 python dict로 변환
    if name not in function_map:
        result = {
            "error" : f"등록하지 않은 함수입니다. : {name}"
        }
    else:
        result = function_map[name](**args)

    # 실행 결과를 모델에게 다시 전달할 function_call_output 형식으로 변환
    return{
        "type" : "function_call_output",
        "call_id" : item.call_id,
        "output" : json.dumps(result,ensure_ascii=False)
    }

def run_with_tools(prompt,tools,function_map,instructions,max_round=5,verbose=True):
    """Function Calling 전체 흐름을 반복 실행한다."""

    input_items = prompt            # 첫 요청에는 사용자 프롬프트를 그대로 전달
    previous_response_id = None     # 이전 응답과 다음 요청을 연결하기 위한 response id

    # 모델이 함수 호출을 여러번 이어갈 수 있으므로 최대 횟수까지 반복
    for round_no in range(1,max_round + 1):

        request_args = {
            "model" : DEFAULT_MODEL,
            "instructions" : instructions,
            "input" : input_items,
            "tools" : tools
        }

        if previous_response_id:
            request_args['previous_response_id'] = previous_response_id

        response = client.responses.create(**request_args)

        previous_response_id = response.id

        if verbose:
            print(f'[rount {round_no}] output types :',[item.type for item in response.output])

        function_outputs = []

        # 모델 응답 중 function_call이 있으면 함수를 호출한다.
        for item in response.output:
            if item.type == "function_call":
                function_outputs.append(run_function_call(item, function_map))

        # functon_call이 없으면 최종 답변이 생성된 것이므로 반환
        if not function_outputs:
            return response.output_text
        
        if verbose:
            print(f"[round {round_no}] function outputs : ")
            for output in function_outputs:
                print(output)

        # 다음 요청에 실행한 함수 결과를 전달
        input_items = function_outputs

    return "최대 반복 횟수에 도달했습니다. 함수 호출 흐름을 확인하세요"

## 수강생 삼담 예제 실행

In [7]:
student_instructions = """
너는 수강생 학습 상담을 돕는 AI 보조 강사다.
필요한 경우 제공된 함수를 사용한다.
학생의 점수, 출석률, 지각 횟수, 상담 피드백 근거를 반영하여 구체적으로 답한다.
"""

answer = run_with_tools(
    prompt="철수 학생의 현재 상태를 보고 상담 코멘트를 작성해줘",
    tools=student_tools,
    function_map=student_function_map,
    instructions=student_instructions
)

print(answer)

[rount 1] output types : ['function_call']
[round 1] function outputs : 
{'type': 'function_call_output', 'call_id': 'call_mGzweELjezBL27FCi9LpQ1Dk', 'output': '{"found": true, "name": "철수", "score": 82, "attendance": 0.92, "late_count": 1, "level": "보통", "message": "기본기는 있으나 오답 유형을 점검하는 보충 수업이 필요합니다."}'}
[rount 2] output types : ['message']
철수 학생은 현재 점수 82점, 출석률 92%, 지각 1회로 전체적으로 보통 수준입니다. 기본기는 잘 갖추고 있으나, 오답 유형에 대해 점검하는 보충 수업이 필요하다는 피드백이 있습니다. 학습에서 틀린 문제 유형을 꼼꼼히 분석하고 보완하는 데 집중한다면 성적 향상에 도움이 될 것입니다. 앞으로 보충 수업과 복습에 힘쓰도록 지도하는 것이 좋겠습니다.
